# Home Credit Default Risk Predictor
## Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

In [2]:
train = pd.read_csv('application_train.csv')
test = pd.read_csv('application_test.csv')

print(f'Training Data Shape : {train.shape}')
print(f'Test Data Shape     : {test.shape}')

Training Data Shape : (307511, 23)
Test Data Shape     : (48744, 22)


---
## 1. Handling Missing Values

From the Data Understanding phase, the following columns have missing values:

| Column | Strategy | Reason |
|---|---|---|
| `EXT_SOURCE_1` | Fill with **Median** | ~56% missing — external credit score, median is robust to outliers |
| `EXT_SOURCE_3` | Fill with **Median** | ~20% missing — external credit score, median is robust to outliers |
| `EXT_SOURCE_2` | Fill with **Median** | <0.2% missing — external credit score |
| `OCCUPATION_TYPE` | Fill with **"Unknown"** | ~31% missing — represents unreported occupation, treated as its own category |
| `NAME_TYPE_SUITE` | Fill with **Mode** | <1% missing — categorical column, mode preserves the dominant distribution |
| `AMT_GOODS_PRICE` | Fill with **Median** | <0.1% missing — numeric column, median is robust to outliers |
| `AMT_ANNUITY` | Fill with **Median** | <0.01% missing — numeric column, median is robust to outliers |

### 1.1 Missing Values Before Imputation

In [3]:
def show_missing(df, name):
    """Display missing value counts and percentages for a DataFrame."""
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if missing.empty:
        print(f'{name}: No missing values found!')
    else:
        missing_df = pd.DataFrame({
            'Missing Count': missing,
            'Missing %': round((missing / len(df)) * 100, 2)
        }).sort_values('Missing Count', ascending=False)
        print(f'--- {name}: Missing Values ---')
        print(missing_df)
        print()

show_missing(train, 'Training Data')
show_missing(test, 'Test Data')

--- Training Data: Missing Values ---
                 Missing Count  Missing %
EXT_SOURCE_1            173378      56.38
OCCUPATION_TYPE          96391      31.35
EXT_SOURCE_3             60965      19.83
NAME_TYPE_SUITE           1292       0.42
EXT_SOURCE_2               660       0.21
AMT_GOODS_PRICE            278       0.09
AMT_ANNUITY                 12       0.00

--- Test Data: Missing Values ---
                 Missing Count  Missing %
EXT_SOURCE_1             20532      42.12
OCCUPATION_TYPE          15605      32.01
EXT_SOURCE_3              8668      17.78
NAME_TYPE_SUITE            911       1.87
AMT_ANNUITY                 24       0.05
EXT_SOURCE_2                 8       0.02



### 1.2 Impute Missing Values

In [4]:
# --- EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3: Fill with Median ---
for col in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']:
    median_val = train[col].median()
    train[col] = train[col].fillna(median_val)
    test[col] = test[col].fillna(median_val)
    print(f'{col} median (from train): {median_val:.6f}')
    print(f'  Train missing after fill: {train[col].isnull().sum()}')
    print(f'  Test  missing after fill: {test[col].isnull().sum()}')

EXT_SOURCE_1 median (from train): 0.505998
  Train missing after fill: 0
  Test  missing after fill: 0
EXT_SOURCE_2 median (from train): 0.565961
  Train missing after fill: 0
  Test  missing after fill: 0
EXT_SOURCE_3 median (from train): 0.535276
  Train missing after fill: 0
  Test  missing after fill: 0


In [5]:
# --- OCCUPATION_TYPE: Fill with "Unknown" ---
train['OCCUPATION_TYPE'] = train['OCCUPATION_TYPE'].fillna('Unknown')
test['OCCUPATION_TYPE'] = test['OCCUPATION_TYPE'].fillna('Unknown')

print(f'OCCUPATION_TYPE (train) missing after fill: {train["OCCUPATION_TYPE"].isnull().sum()}')
print(f'OCCUPATION_TYPE (test)  missing after fill: {test["OCCUPATION_TYPE"].isnull().sum()}')

OCCUPATION_TYPE (train) missing after fill: 0
OCCUPATION_TYPE (test)  missing after fill: 0


In [6]:
# --- NAME_TYPE_SUITE: Fill with Mode ---
# Compute mode from training data and apply to both sets for consistency
suite_mode = train['NAME_TYPE_SUITE'].mode()[0]
print(f'NAME_TYPE_SUITE mode (from train): "{suite_mode}"')

train['NAME_TYPE_SUITE'] = train['NAME_TYPE_SUITE'].fillna(suite_mode)
test['NAME_TYPE_SUITE'] = test['NAME_TYPE_SUITE'].fillna(suite_mode)

print(f'NAME_TYPE_SUITE (train) missing after fill: {train["NAME_TYPE_SUITE"].isnull().sum()}')
print(f'NAME_TYPE_SUITE (test)  missing after fill: {test["NAME_TYPE_SUITE"].isnull().sum()}')

NAME_TYPE_SUITE mode (from train): "Unaccompanied"
NAME_TYPE_SUITE (train) missing after fill: 0
NAME_TYPE_SUITE (test)  missing after fill: 0


In [7]:
# --- AMT_GOODS_PRICE: Fill with Median ---
goods_price_median = train['AMT_GOODS_PRICE'].median()
print(f'AMT_GOODS_PRICE median (from train): {goods_price_median}')

train['AMT_GOODS_PRICE'] = train['AMT_GOODS_PRICE'].fillna(goods_price_median)
test['AMT_GOODS_PRICE'] = test['AMT_GOODS_PRICE'].fillna(goods_price_median)

print(f'AMT_GOODS_PRICE (train) missing after fill: {train["AMT_GOODS_PRICE"].isnull().sum()}')
print(f'AMT_GOODS_PRICE (test)  missing after fill: {test["AMT_GOODS_PRICE"].isnull().sum()}')

AMT_GOODS_PRICE median (from train): 450000.0
AMT_GOODS_PRICE (train) missing after fill: 0
AMT_GOODS_PRICE (test)  missing after fill: 0


In [8]:
# --- AMT_ANNUITY: Fill with Median ---
annuity_median = train['AMT_ANNUITY'].median()
print(f'AMT_ANNUITY median (from train): {annuity_median}')

train['AMT_ANNUITY'] = train['AMT_ANNUITY'].fillna(annuity_median)
test['AMT_ANNUITY'] = test['AMT_ANNUITY'].fillna(annuity_median)

print(f'AMT_ANNUITY (train) missing after fill: {train["AMT_ANNUITY"].isnull().sum()}')
print(f'AMT_ANNUITY (test)  missing after fill: {test["AMT_ANNUITY"].isnull().sum()}')

AMT_ANNUITY median (from train): 24903.0
AMT_ANNUITY (train) missing after fill: 0
AMT_ANNUITY (test)  missing after fill: 0


### 1.3 Verify — No Missing Values Remain

In [9]:
show_missing(train, 'Training Data')
show_missing(test, 'Test Data')

Training Data: No missing values found!
Test Data: No missing values found!


---
## 2. Feature Transformations

Convert raw `DAYS_BIRTH` and `DAYS_EMPLOYED` into more interpretable features.

| Original Column | New Column | Transformation |
|---|---|---|
| `DAYS_BIRTH` | `AGE` | `abs(DAYS_BIRTH) / 365.25` — age in years |
| `DAYS_EMPLOYED` | `YEARS_EMPLOYED` | Replace anomaly (365243) with 0, then `abs(DAYS_EMPLOYED) / 365.25` |

In [10]:
# --- DAYS_BIRTH → AGE ---
train['AGE'] = (train['DAYS_BIRTH'].abs() / 365.25).round(2)
test['AGE'] = (test['DAYS_BIRTH'].abs() / 365.25).round(2)

# Drop original column
train = train.drop(columns=['DAYS_BIRTH'])
test = test.drop(columns=['DAYS_BIRTH'])

print('AGE — Sample values (train):')
print(train['AGE'].describe())

AGE — Sample values (train):
count    307511.000000
mean         43.906915
std          11.947952
min          20.500000
25%          33.980000
50%          43.120000
75%          53.890000
max          69.070000
Name: AGE, dtype: float64


In [11]:
# --- DAYS_EMPLOYED → YEARS_EMPLOYED ---
# Step 1: Handle the anomalous value 365243 (represents ~1000 years — used for unemployed/pensioners)
anomaly_value = 365243
for df in [train, test]:
    anomaly_count = (df['DAYS_EMPLOYED'] == anomaly_value).sum()
    print(f'Anomalous DAYS_EMPLOYED values: {anomaly_count}')
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(anomaly_value, 0)

# Step 2: Convert to years
train['YEARS_EMPLOYED'] = (train['DAYS_EMPLOYED'].abs() / 365.25).round(2)
test['YEARS_EMPLOYED'] = (test['DAYS_EMPLOYED'].abs() / 365.25).round(2)

# Drop original column
train = train.drop(columns=['DAYS_EMPLOYED'])
test = test.drop(columns=['DAYS_EMPLOYED'])

print('\nYEARS_EMPLOYED — Sample values (train):')
print(train['YEARS_EMPLOYED'].describe())

Anomalous DAYS_EMPLOYED values: 55374
Anomalous DAYS_EMPLOYED values: 9274

YEARS_EMPLOYED — Sample values (train):
count    307511.000000
mean          5.352081
std           6.316409
min           0.000000
25%           0.790000
50%           3.320000
75%           7.560000
max          49.040000
Name: YEARS_EMPLOYED, dtype: float64


---
## 3. Feature Encoding

Now that missing values are handled and transformations are done, we encode categorical features into numeric form.

| Encoding | Columns | Reason |
|---|---|---|
| **Label Encoding** | `NAME_CONTRACT_TYPE`, `CODE_GENDER`, `FLAG_OWN_CAR`, `FLAG_OWN_REALTY` | Binary / low-cardinality nominal features |
| **Ordinal Encoding** | `NAME_EDUCATION_TYPE` | Natural order exists among education levels |
| **One-Hot Encoding** | `NAME_TYPE_SUITE`, `NAME_INCOME_TYPE`, `NAME_FAMILY_STATUS`, `OCCUPATION_TYPE`, `NAME_HOUSING_TYPE` | Multi-class nominal features with no inherent order |
| **Target Encoding** | `ORGANIZATION_TYPE` | 58 categories — too many for one-hot; encode with mean TARGET rate from training data |

### 3.1 Label Encoding

These columns are binary or have only 2–3 categories, so a single integer column suffices.

- `NAME_CONTRACT_TYPE` → Cash loans / Revolving loans
- `CODE_GENDER` → M / F
- `FLAG_OWN_CAR` → Y / N
- `FLAG_OWN_REALTY` → Y / N

In [12]:
label_encode_cols = ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY']

label_encoders = {}

for col in label_encode_cols:
    le = LabelEncoder()
    # Fit on training data
    train[col] = le.fit_transform(train[col])
    # Transform test data using the same encoder
    test[col] = le.transform(test[col])
    label_encoders[col] = le
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

NAME_CONTRACT_TYPE: {'Cash loans': np.int64(0), 'Revolving loans': np.int64(1)}
CODE_GENDER: {'F': np.int64(0), 'M': np.int64(1), 'XNA': np.int64(2)}
FLAG_OWN_CAR: {'N': np.int64(0), 'Y': np.int64(1)}
FLAG_OWN_REALTY: {'N': np.int64(0), 'Y': np.int64(1)}


In [13]:
# Quick check
train[label_encode_cols].head()

,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY
0,0,1,0,1
1,0,0,0,0
2,1,1,1,1
3,0,0,0,1
4,0,1,0,1


### 3.2 Ordinal Encoding

`NAME_EDUCATION_TYPE` has a natural hierarchy:

| Rank | Education Level |
|---|---|
| 0 | Lower secondary |
| 1 | Secondary / secondary special |
| 2 | Incomplete higher |
| 3 | Higher education |
| 4 | Academic degree |

In [14]:
education_order = [
    'Lower secondary',
    'Secondary / secondary special',
    'Incomplete higher',
    'Higher education',
    'Academic degree'
]

ordinal_encoder = OrdinalEncoder(categories=[education_order])

train['NAME_EDUCATION_TYPE'] = ordinal_encoder.fit_transform(train[['NAME_EDUCATION_TYPE']]).astype(int)
test['NAME_EDUCATION_TYPE'] = ordinal_encoder.transform(test[['NAME_EDUCATION_TYPE']]).astype(int)

print('Ordinal Encoding Mapping:')
for i, level in enumerate(education_order):
    print(f'  {i} → {level}')

print(f'\nUnique values in train: {sorted(train["NAME_EDUCATION_TYPE"].unique())}')
print(f'Unique values in test : {sorted(test["NAME_EDUCATION_TYPE"].unique())}')

Ordinal Encoding Mapping:
  0 → Lower secondary
  1 → Secondary / secondary special
  2 → Incomplete higher
  3 → Higher education
  4 → Academic degree

Unique values in train: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Unique values in test : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


### 3.3 Target Encoding — ORGANIZATION_TYPE

`ORGANIZATION_TYPE` has 58 unique values — too many for one-hot encoding (would add 57 sparse columns).

We use **target encoding**: replace each category with the mean `TARGET` rate from the **training set only**.

This creates a single numeric column that captures the default risk associated with each organization type.

In [16]:
# Compute mean TARGET per ORGANIZATION_TYPE from training data
org_target_mean = train.groupby('ORGANIZATION_TYPE')['TARGET'].mean()

print(f'ORGANIZATION_TYPE: {len(org_target_mean)} unique values')
print(f'\nTarget encoding mapping (top 10 by default rate):')
for org, rate in org_target_mean.sort_values(ascending=False).head(10).items():
    print(f'  {org:<35} → {rate:.4f}')

# Apply target encoding to both train and test
# For any unseen categories in test, use the global mean
global_mean = train['TARGET'].mean()
train['ORGANIZATION_TYPE'] = train['ORGANIZATION_TYPE'].map(org_target_mean)
test['ORGANIZATION_TYPE'] = test['ORGANIZATION_TYPE'].map(org_target_mean).fillna(global_mean)

print(f'\nORGANIZATION_TYPE (train) — dtype: {train["ORGANIZATION_TYPE"].dtype}')
print(f'ORGANIZATION_TYPE (test)  — dtype: {test["ORGANIZATION_TYPE"].dtype}')
print(f'Train NaN: {train["ORGANIZATION_TYPE"].isnull().sum()}, Test NaN: {test["ORGANIZATION_TYPE"].isnull().sum()}')

ORGANIZATION_TYPE: 57 unique values

Target encoding mapping (top 10 by default rate):
  0.15754001684919966                 → 0.1575
  0.13432835820895522                 → 0.1343
  0.125                               → 0.1250
  0.11706239646604086                 → 0.1171
  0.11679809552149978                 → 0.1168
  0.11153846153846154                 → 0.1115
  0.11068334937439846                 → 0.1107
  0.10616229408175717                 → 0.1062
  0.10606060606060606                 → 0.1061
  0.10472697636511817                 → 0.1047

ORGANIZATION_TYPE (train) — dtype: float64
ORGANIZATION_TYPE (test)  — dtype: float64
Train NaN: 0, Test NaN: 0


### 3.4 One-Hot Encoding

For multi-class nominal features with no inherent ordering, we apply one-hot encoding.

- `NAME_TYPE_SUITE`
- `NAME_INCOME_TYPE`
- `NAME_FAMILY_STATUS`
- `OCCUPATION_TYPE`
- `NAME_HOUSING_TYPE`

We use `pd.get_dummies()` and align train/test columns to handle any categories that appear in only one set.

In [17]:
one_hot_cols = ['NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_FAMILY_STATUS', 'OCCUPATION_TYPE',
                'NAME_HOUSING_TYPE']

print('Cardinality before one-hot encoding:')
for col in one_hot_cols:
    print(f'  {col}: {train[col].nunique()} unique values')

Cardinality before one-hot encoding:
  NAME_TYPE_SUITE: 7 unique values
  NAME_INCOME_TYPE: 8 unique values
  NAME_FAMILY_STATUS: 6 unique values
  OCCUPATION_TYPE: 19 unique values
  NAME_HOUSING_TYPE: 6 unique values


In [18]:
# Apply one-hot encoding
train = pd.get_dummies(train, columns=one_hot_cols, drop_first=True, dtype=int)
test = pd.get_dummies(test, columns=one_hot_cols, drop_first=True, dtype=int)

# Align train and test columns (in case some categories only exist in one set)
# Keep TARGET column safe — it only exists in train
train_cols = set(train.columns)
test_cols = set(test.columns)

# Add missing columns to test (filled with 0)
for col in train_cols - test_cols:
    if col != 'TARGET':
        test[col] = 0

# Add missing columns to train (filled with 0)
for col in test_cols - train_cols:
    train[col] = 0

# Reorder test columns to match train (excluding TARGET)
test = test[[c for c in train.columns if c != 'TARGET']]

print(f'Training Data Shape after encoding : {train.shape}')
print(f'Test Data Shape after encoding     : {test.shape}')

Training Data Shape after encoding : (307511, 59)
Test Data Shape after encoding     : (48744, 58)


In [19]:
# Verify: all columns should now be numeric
print('--- Training Data Types ---')
print(train.dtypes.value_counts())
print()

non_numeric = train.select_dtypes(include='object').columns.tolist()
if non_numeric:
    print(f'WARNING: Non-numeric columns remaining: {non_numeric}')
else:
    print('All columns are numeric. Ready for modeling!')

--- Training Data Types ---
int64      49
float64    10
Name: count, dtype: int64

All columns are numeric. Ready for modeling!


In [20]:
train.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,OCCUPATION_TYPE_Sales staff,OCCUPATION_TYPE_Secretaries,OCCUPATION_TYPE_Security staff,OCCUPATION_TYPE_Unknown,OCCUPATION_TYPE_Waiters/barmen staff,NAME_HOUSING_TYPE_House / apartment,NAME_HOUSING_TYPE_Municipal apartment,NAME_HOUSING_TYPE_Office apartment,NAME_HOUSING_TYPE_Rented apartment,NAME_HOUSING_TYPE_With parents
0,100002,1,0,1,0,1,0,202500.0,406597.5,24700.5,...,0,0,0,0,0,1,0,0,0,0
1,100003,0,0,0,0,0,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0,1,0,0,0,0
2,100004,0,1,1,1,1,0,67500.0,135000.0,6750.0,...,0,0,0,0,0,1,0,0,0,0
3,100006,0,0,0,0,1,0,135000.0,312682.5,29686.5,...,0,0,0,0,0,1,0,0,0,0
4,100007,0,0,1,0,1,0,121500.0,513000.0,21865.5,...,0,0,0,0,0,1,0,0,0,0


---
## 4. Summary

| Step | Action | Details |
|---|---|---|
| Missing Values | `EXT_SOURCE_1/2/3` → Median | External credit scores imputed with training median |
| | `OCCUPATION_TYPE` → "Unknown" | 31% missing treated as a separate category |
| | `NAME_TYPE_SUITE` → Mode | <1% missing filled with most frequent value |
| | `AMT_GOODS_PRICE` → Median | Robust to outliers |
| | `AMT_ANNUITY` → Median | Robust to outliers |
| Transformations | `DAYS_BIRTH` → `AGE` | Converted to years |
| | `DAYS_EMPLOYED` → `YEARS_EMPLOYED` | Anomaly (365243) replaced with 0, then converted to years |
| Label Encoding | 4 binary columns | `NAME_CONTRACT_TYPE`, `CODE_GENDER`, `FLAG_OWN_CAR`, `FLAG_OWN_REALTY` |
| Ordinal Encoding | `NAME_EDUCATION_TYPE` | 5-level hierarchy (Lower secondary → Academic degree) |
| Target Encoding | `ORGANIZATION_TYPE` | 58 categories → mean TARGET rate from training data |
| One-Hot Encoding | 5 nominal columns | `NAME_TYPE_SUITE`, `NAME_INCOME_TYPE`, `NAME_FAMILY_STATUS`, `OCCUPATION_TYPE`, `NAME_HOUSING_TYPE` |

In [21]:
# Save preprocessed data
train.to_csv('train_preprocessed.csv', index=False)
test.to_csv('test_preprocessed.csv', index=False)

print('Preprocessed data saved!')
print(f'  train_preprocessed.csv : {train.shape}')
print(f'  test_preprocessed.csv  : {test.shape}')

Preprocessed data saved!
  train_preprocessed.csv : (307511, 59)
  test_preprocessed.csv  : (48744, 58)
